# XGBoost and LightBoost

## Introduction

After developing a detailed, hands-on understanding of gradient boosting using the scikit-learn implementation, the next step is to evaluate **specialized boosting frameworks** widely used in industry, namely XGBoost and LightGBM.

These libraries are designed to deliver high performance on structured data with significantly less manual tuning. They incorporate a range of algorithmic and system-level optimizations, including efficient tree construction, built-in regularization, and advanced handling of large feature spaces. As a result, they often achieve strong predictive performance with relatively modest configuration effort.

In contrast to the previous chapter, which emphasized a **hands-on and controlled modeling approach**, this section adopts a more **pragmatic, model-driven workflow**. Rather than carefully shaping model behavior through extensive hyperparameter tuning, the focus shifts toward leveraging the strengths of these frameworks and allowing them to operate with largely default or lightly tuned settings.

This approach reflects a common real-world scenario, where the goal is not to fully explore the parameter space, but to obtain a strong and reliable model efficiently using well-optimized tools. In this sense, the modeling process can be viewed as a more **“hands-off” approach**, where the underlying algorithms handle much of the complexity.

The objective of this chapter is therefore twofold:

- to evaluate whether these specialized implementations improve predictive performance compared to the scikit-learn gradient boosting model
- to assess the trade-off between manual control and practical efficiency

By comparing results across these approaches, we gain a clearer understanding of when it is beneficial to rely on highly optimized external libraries versus when a more transparent and controlled modeling process is preferred.

## Table of Contents

1. [Modeling Approach](#modeling-approach)
2. [XGboost](#xgboost)
3. [LightBoost](#lightgbm)
4. [Executive Summary](#executive-summary)

## Modeling Approach

In contrast to the previous chapter, which followed a structured and iterative modeling process, this section adopts a more **streamlined and pragmatic approach** using specialized gradient boosting libraries.

Rather than performing extensive hyperparameter tuning and multi-stage model refinement, the workflow is simplified to reflect a more practical, production-oriented setting. The goal is to evaluate how well modern boosting frameworks perform with minimal manual intervention.

The modeling process is therefore reduced to three main steps:

1. **Train the model** on the training dataset using default or lightly specified parameters
2. **Evaluate performance** on the validation set to assess generalization and adjust only if necessary
3. **Apply the final model** to the hold-out test set for unbiased performance estimation

This approach emphasizes ease of use and efficiency, leveraging the strong default configurations and built-in optimizations provided by libraries such as XGBoost and LightGBM.

Unlike the previous chapter, where the focus was on understanding model behavior through careful tuning, the objective here is to assess how these tools perform when used in a more **plug-and-play manner**. This reflects a common real-world scenario, where time constraints or system requirements favor fast, reliable model deployment over exhaustive parameter optimization.

The results from this approach will be compared directly with the previously developed gradient boosting model to evaluate the trade-off between manual control and automated performance.

This chapter intentionally shifts from a detailed, parameter-driven workflow to a more lightweight and practical approach, relying on the strength of modern boosting libraries to deliver strong performance with minimal manual tuning.


Next step, data preparation:

In [1]:
import polars as pl
from sklearn.model_selection import train_test_split

# Load datasets
train_df = pl.read_csv("./data/processed/06_dpp_train_df.csv")
test_df = pl.read_csv("./data/processed/06_dpp_test_df.csv")

selected_cols = [
    'SeniorCitizenRelevel',
    'Partner',
    'Dependents',
    'tenure',
    'Contract',
    'PaperlessBilling',
    'MonthlyCharges',
    'Churn',
    'AdditionalInternetServicesCount',
    'StreamingServicesCount',
    'PaymentMethod_bin_3'
]

train_df = train_df.select(selected_cols)
test_df = test_df.select(selected_cols)

# Convert to pandas for sklearn
train_pd = train_df.to_pandas()

# Stratified split: 25% of current train -> validation
train_sub_pd, val_pd = train_test_split(
    train_pd,
    test_size=0.25,
    stratify=train_pd["Churn"],
    random_state=42
)

# Back to polars
train_sub_df = pl.from_pandas(train_sub_pd)
val_df = pl.from_pandas(val_pd)

# Shapes
print("Train subset shape:", train_sub_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

# Class proportions helper
def class_proportions(df, name):
    print(f"\n{name} class proportions:")
    print(
        df.group_by("Churn")
        .len()
        .with_columns(
            (pl.col("len") / pl.col("len").sum()).alias("proportion")
        )
        .sort("Churn")
    )

class_proportions(train_sub_df, "Train subset")
class_proportions(val_df, "Validation")
class_proportions(test_df, "Test")

Train subset shape: (4225, 11)
Validation shape: (1409, 11)
Test shape: (1409, 11)

Train subset class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 3104 ┆ 0.734675   │
│ Yes   ┆ 1121 ┆ 0.265325   │
└───────┴──────┴────────────┘

Validation class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 1035 ┆ 0.734564   │
│ Yes   ┆ 374  ┆ 0.265436   │
└───────┴──────┴────────────┘

Test class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 1035 ┆ 0.734564   │
│ Yes   ┆ 374  ┆ 0.265436   │
└───────┴──────┴────────────┘


In [2]:
from src.utils.data_preparation_models import prepare_tree_features

categorical_cols = ['SeniorCitizenRelevel',
                    'Partner',
                    'Dependents',
                    'Contract',
                    'PaperlessBilling',
                    'PaymentMethod_bin_3']

target_col = "Churn"

numerical_cols = ['tenure', 
                  'MonthlyCharges', 
                  'AdditionalInternetServicesCount', 
                  'StreamingServicesCount']   

categorical_orders = { "SeniorCitizenRelevel": ["No", "Yes"], 
                      "Partner": ["No", "Yes"], 
                      "Dependents": ["No", "Yes"], 
                      "PaperlessBilling": ["No", "Yes"], 
                      "Contract": ["Month-to-month", "One year", "Two year"] }

train_X, train_y = prepare_tree_features(
    df=train_sub_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    target_col=target_col
)

validation_X, validation_y = prepare_tree_features(
    df=val_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    target_col=target_col,
    reference_columns=train_X.columns.tolist()
)

For consistency and comparability with previous models, the same feature representation is used in this section. Although libraries such as XGBoost and LightGBM provide native handling of categorical variables, one-hot encoded features are retained to ensure that differences in performance can be attributed to the modeling approach rather than changes in preprocessing.

## XGBoost

XGBoost (Extreme Gradient Boosting) is a highly optimized implementation of gradient boosting designed for performance, scalability, and practical use in real-world machine learning tasks. It extends the standard gradient boosting framework by introducing both algorithmic improvements and system-level optimizations.

At its core, XGBoost follows the same principle as other boosting methods: it builds an ensemble of decision trees sequentially, where each new tree is trained to correct the errors of the existing model. However, it enhances this process through several key features.

First, XGBoost includes **regularization directly in the learning objective**, helping to control model complexity and reduce overfitting. This is particularly important for boosting models, which can otherwise become overly sensitive to noise in the data.

Second, it uses **second-order optimization**, incorporating both gradients and Hessians when updating the model. This allows for more accurate and stable learning compared to traditional gradient boosting implementations.

Third, XGBoost is designed with performance in mind. It supports efficient tree construction, parallel processing, and optimized memory usage, making it suitable for both small and large-scale datasets.

---

### Modeling Perspective

In contrast to the structured and iterative tuning approach used in the previous chapter, XGBoost is applied here in a more **pragmatic and streamlined manner**. The goal is not to exhaustively explore the hyperparameter space, but rather to leverage the strong default configuration of the library to obtain a competitive model with minimal manual intervention.

This reflects a common real-world workflow, where time and computational constraints favor efficient model deployment over extensive tuning. Instead of carefully shaping model behavior through multiple rounds of refinement, the model is trained with largely default settings and evaluated directly on validation data.

The focus is therefore on:

* assessing baseline performance with minimal configuration
* comparing results with the previously developed gradient boosting model
* understanding the trade-off between manual control and automated optimization

---

### Key Characteristics

XGBoost introduces several practical advantages over standard implementations:

* **Built-in regularization** helps prevent overfitting
* **Efficient tree construction** improves training speed
* **Handling of sparse data** allows flexibility with feature representations
* **Strong default performance** often reduces the need for extensive tuning

At the same time, these benefits come with reduced transparency. As with other boosting models, interpretability is limited, and the model behaves as a complex ensemble rather than a set of explicit decision rules.

---

Overall, XGBoost represents a powerful and practical extension of gradient boosting, offering strong predictive performance with a simplified modeling workflow. In this section, it is evaluated in a plug-and-play setting to determine how effectively it performs relative to the previously developed model.


In [3]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"   # avoids warning
)

xgb_model.fit(train_X, train_y)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [4]:
import pandas as pd
from src.utils.classification_metrics import compute_classification_metrics

xgb_val_prob = xgb_model.predict_proba(validation_X)[:, 1]
xgb_val_pred = (xgb_val_prob >= 0.5).astype(int)

xgb_val_metrics = compute_classification_metrics(
    y_true=validation_y,
    y_pred=xgb_val_pred,
    y_prob=xgb_val_prob
)


pd.DataFrame([xgb_val_metrics])


,accuracy,precision,recall,f1,auc,pr_auc,tp,tn,fp,fn
0,0.757984,0.554817,0.446524,0.494815,0.78773,0.556987,167,901,134,207


The XGBoost model was trained using default parameters and evaluated on the validation set to assess its performance in a plug-and-play setting.

The model achieves an F1 score of 0.4948, reflecting the balance between precision (0.5548) and recall (0.4465). Compared to the previously developed gradient boosting model, this represents a noticeable decrease in classification performance, particularly in recall, indicating that fewer churners are correctly identified.

From a ranking perspective, the model achieves a ROC-AUC of 0.7877 and a PR-AUC of 0.5570. These values are noticeably lower than those observed for the HistGradientBoosting model, suggesting weaker ability to distinguish between churners and non-churners and to prioritize high-risk customers.

The confusion matrix further highlights this behavior, with 167 true positives and 207 false negatives, indicating that a significant portion of churners is not captured by the model.

Overall, while XGBoost is known for strong performance, the default configuration in this case does not match the performance of the more carefully tuned gradient boosting model. This highlights the importance of hyperparameter tuning, even when using highly optimized external libraries.

These results suggest that, in this setting, a fully hands-off approach may not be sufficient to achieve optimal performance.

## LightGBM

LightGBM (Light Gradient Boosting Machine) is a highly efficient gradient boosting framework developed by Microsoft. It is designed to deliver fast training, low memory usage, and strong predictive performance, particularly on large-scale and high-dimensional datasets.

Like other boosting methods, LightGBM builds an ensemble of decision trees sequentially, where each new tree focuses on correcting the errors of the existing model. However, it introduces several key innovations that differentiate it from traditional gradient boosting implementations.

One of the main features of LightGBM is its **leaf-wise tree growth strategy**. Unlike level-wise growth used in many other implementations, LightGBM expands the tree by splitting the leaf that leads to the largest reduction in loss. This allows the model to achieve higher accuracy with fewer splits, but also increases the risk of overfitting if not properly controlled.

In addition, LightGBM uses a **histogram-based algorithm** similar to the one used in scikit-learn’s histogram gradient boosting. Continuous features are bucketed into discrete bins, which significantly improves training speed and reduces memory consumption.

---

### Modeling Perspective

As with XGBoost, LightGBM is applied in this section using a **streamlined, plug-and-play approach**. The goal is to evaluate how well the model performs with minimal manual tuning, leveraging its strong default configuration and built-in optimizations.

Rather than performing extensive hyperparameter search, the model is trained directly on the training data and evaluated on the validation set. This reflects a practical scenario where rapid model development and deployment are prioritized.

The focus is therefore on:

* assessing baseline performance with minimal configuration
* comparing results with both the tuned gradient boosting model and XGBoost
* evaluating whether LightGBM can achieve competitive performance without detailed tuning

---

### Key Characteristics

LightGBM offers several advantages that make it widely used in practice:

* **Leaf-wise tree growth** for potentially higher accuracy
* **Efficient histogram-based training** for speed and scalability
* **Low memory usage**, making it suitable for large datasets
* **Strong default performance**, often requiring minimal tuning

However, these benefits come with similar trade-offs to other boosting methods. The model is less interpretable than simpler approaches, and its aggressive tree growth strategy can lead to overfitting if not carefully managed.

---

Overall, LightGBM provides a powerful and efficient implementation of gradient boosting. In this section, it is evaluated under a minimal-configuration setting to determine how it performs relative to both XGBoost and the previously tuned gradient boosting model.


In [5]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    random_state=42
)

lgbm_model.fit(
    train_X,
    train_y,
    eval_set=[(validation_X, validation_y)],
    eval_metric="logloss"
)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1121, number of negative: 3104
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000446 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 364
[LightGBM] [Info] Number of data points in the train set: 4225, number of used features: 18
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265325 -> initscore=-1.018470
[LightGBM] [Info] Start training from score -1.018470


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


Validate on validation set:

In [6]:
from src.utils.classification_metrics import compute_classification_metrics

lgbm_val_prob = lgbm_model.predict_proba(validation_X)[:, 1]
lgbm_val_pred = (lgbm_val_prob >= 0.5).astype(int)

lgbm_val_metrics = compute_classification_metrics(
    y_true=validation_y,
    y_pred=lgbm_val_pred,
    y_prob=lgbm_val_prob
)

import pandas as pd
pd.DataFrame([lgbm_val_metrics])

,accuracy,precision,recall,f1,auc,pr_auc,tp,tn,fp,fn
0,0.789922,0.634483,0.491979,0.554217,0.805187,0.605357,184,929,106,190


### Validation Results — LightGBM

The LightGBM model was trained using default parameters and evaluated on the validation set to assess its performance in a plug-and-play setting.

The model achieves an F1 score of **0.5542**, reflecting a balanced trade-off between precision (**0.6345**) and recall (**0.4920**). Compared to XGBoost, this represents a substantial improvement in both recall and overall classification performance, indicating that LightGBM is more effective at identifying churners in this configuration.

From a ranking perspective, the model achieves a ROC-AUC of **0.8052** and a PR-AUC of **0.6054**. While these values remain below those achieved by the tuned gradient boosting model, they are significantly higher than those observed for XGBoost, demonstrating stronger ability to distinguish between churners and non-churners.

The confusion matrix shows that the model correctly identifies **184 churners** while missing **190**, indicating improved recall relative to XGBoost, with a moderate increase in false positives.

Overall, LightGBM delivers strong performance with minimal configuration, approaching the performance of the carefully tuned gradient boosting model. This highlights its effectiveness as a practical, high-performing solution in settings where extensive hyperparameter tuning is not feasible.


## Retraining the model on entire dataset

After comparing model performance on the validation set, the selected models are retrained on the combined training and validation data. This ensures that all available development data is used for parameter estimation before final evaluation.

The retrained models are then applied to the hold-out test set, which has remained untouched throughout model development. This provides an unbiased estimate of out-of-sample performance and allows direct comparison of the final plug-and-play boosting models.

In [7]:
import numpy as np
import pandas as pd

full_train_X = pd.concat([train_X, validation_X], axis=0).reset_index(drop=True)
full_train_y = np.concatenate([train_y, validation_y], axis=0)

print(full_train_X.shape)
print(full_train_y.shape)

(5634, 18)
(5634,)


In [8]:
from xgboost import XGBClassifier

xgb_final = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

xgb_final.fit(full_train_X, full_train_y)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [9]:
from lightgbm import LGBMClassifier

lgbm_final = LGBMClassifier(
    random_state=42
)

lgbm_final.fit(
    train_X,
    train_y,
    eval_set=[(full_train_X, full_train_y)],
    eval_metric="logloss"
)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1121, number of negative: 3104
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000182 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 364
[LightGBM] [Info] Number of data points in the train set: 4225, number of used features: 18
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265325 -> initscore=-1.018470
[LightGBM] [Info] Start training from score -1.018470


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [10]:
test_X, test_y = prepare_tree_features(
    df=test_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    target_col=target_col,
    reference_columns=train_X.columns.tolist()
)

In [11]:
from src.utils.classification_metrics import compute_classification_metrics
import pandas as pd

# XGBoost test evaluation
xgb_test_prob = xgb_final.predict_proba(test_X)[:, 1]
xgb_test_pred = (xgb_test_prob >= 0.5).astype(int)

xgb_test_metrics = compute_classification_metrics(
    y_true=test_y,
    y_pred=xgb_test_pred,
    y_prob=xgb_test_prob
)

# LightGBM test evaluation
lgbm_test_prob = lgbm_final.predict_proba(test_X)[:, 1]
lgbm_test_pred = (lgbm_test_prob >= 0.5).astype(int)

lgbm_test_metrics = compute_classification_metrics(
    y_true=test_y,
    y_pred=lgbm_test_pred,
    y_prob=lgbm_test_prob
)

test_comparison = pd.DataFrame([
    {"model": "XGBoost", **xgb_test_metrics},
    {"model": "LightGBM", **lgbm_test_metrics}
]).round(4)

test_comparison

,model,accuracy,precision,recall,f1,auc,pr_auc,tp,tn,fp,fn
0,XGBoost,0.7864,0.6151,0.5214,0.5644,0.8219,0.6369,195,913,122,179
1,LightGBM,0.7871,0.6142,0.5321,0.5702,0.8275,0.6480,199,910,125,175


### Test Set Results — XGBoost vs LightGBM

Both XGBoost and LightGBM demonstrate solid performance on the hold-out test set, with broadly similar results across all evaluation metrics.

LightGBM slightly outperforms XGBoost across most key metrics. It achieves a higher F1 score (**0.5702** vs **0.5644**), driven primarily by improved recall (**0.5321** vs **0.5214**), indicating that it is more effective at identifying churners. This comes at a minimal cost to precision, which remains nearly identical between the two models (**0.6142** vs **0.6151**).

From a ranking perspective, LightGBM also shows stronger performance, with a higher ROC-AUC (**0.8275** vs **0.8219**) and PR-AUC (**0.6480** vs **0.6369**). This suggests a better ability to distinguish between churners and non-churners and to prioritize high-risk customers.

The confusion matrices further support this observation. LightGBM correctly identifies **199 churners** compared to **195** for XGBoost, while maintaining a similar number of false positives (125 vs 122). This reflects a slightly more favorable balance between sensitivity and precision.

Overall, while both models perform comparably, LightGBM provides a consistent and modest improvement across multiple metrics. This reinforces its effectiveness as a strong, out-of-the-box boosting model in a plug-and-play setting.

### Model Persistence

To ensure reproducibility and enable future inference, the trained models are stored together with the data preparation configuration. This includes feature definitions and ordering, which are required to correctly transform new input data.

Each model is serialized using pickle, allowing it to be reloaded and applied without retraining. By packaging both the model and preprocessing metadata, the full prediction pipeline can be reconstructed in a consistent and reliable manner.


In [12]:
import pickle

xgb_model_package = {
    # model
    "model": xgb_final,
    # data preparation setup
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "categorical_orders": categorical_orders,
    "target_col": target_col,
    "reference_columns": train_X.columns.tolist()
}

with open("./models/xgboost_final.pkl", "wb") as f:
    pickle.dump(xgb_model_package, f)

In [13]:
import pickle

lgbm_model_package = {
    # model
    "model": lgbm_final,
    # data preparation setup
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "categorical_orders": categorical_orders,
    "target_col": target_col,
    "reference_columns": train_X.columns.tolist()
}

with open("./models/lightgbm_final.pkl", "wb") as f:
    pickle.dump(lgbm_model_package, f)

## Executive Summary

This section evaluated two specialized gradient boosting frameworks, XGBoost and LightGBM, using a streamlined, plug-and-play modeling approach. In contrast to the previous chapter, which focused on careful hyperparameter tuning and controlled model development, the objective here was to assess how well these libraries perform with minimal manual intervention.

Both models were trained using the same feature representation and evaluated on a validation set, followed by final retraining on the combined training and validation data and evaluation on a hold-out test set. This ensured a fair comparison and an unbiased estimate of generalization performance.

The results highlight clear differences between the two approaches. XGBoost, when used with default parameters, delivered reasonable performance but underperformed relative to the tuned gradient boosting model developed previously. In contrast, LightGBM demonstrated strong out-of-the-box performance, achieving results close to the tuned model without requiring extensive hyperparameter optimization.

On the test set, LightGBM consistently outperformed XGBoost across key metrics. It achieved higher recall (**0.5321** vs **0.5214**) and F1 score (**0.5702** vs **0.5644**), while maintaining comparable precision. In addition, LightGBM showed stronger ranking performance, with higher ROC-AUC (**0.8275**) and PR-AUC (**0.6480**), indicating improved ability to identify and prioritize high-risk customers.

These results illustrate an important trade-off in model development. While a carefully tuned model can achieve the highest performance, modern gradient boosting frameworks—particularly LightGBM—are capable of delivering competitive results with significantly less effort. This makes them highly effective in practical settings where rapid development and deployment are prioritized.

Overall, this analysis demonstrates that while hands-on tuning provides maximum control and performance, a plug-and-play approach using optimized libraries can achieve strong and reliable results with minimal complexity.
